In [2]:
!pip install tensorflow scikit-learn


[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: C:\Users\LEGION\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [3]:
!pip install tensorflow scikit-learn


[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: C:\Users\LEGION\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [4]:
import sys
print(sys.executable)

c:\Users\LEGION\AppData\Local\Programs\Python\Python310\python.exe


In [8]:
!C:\Users\LEGION\AppData\Local\Programs\Python\Python310\python.exe -m pip install librosa


[notice] A new release of pip available: 22.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import librosa
print("librosa working")

librosa working


In [9]:
import sys
!{sys.executable} -m pip install librosa


[notice] A new release of pip available: 22.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import sys
!{sys.executable} -m pip install tensorflow


[notice] A new release of pip available: 22.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:

import os

import numpy as np
import librosa
import warnings
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings('ignore')

def extract_mel_spectrogram(y, sr):
    """
    Converts audio into a 2D image-like representation of frequencies over time.
    """
    # Create Mel-spectrogram
    mel_spec = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=2048, hop_length=512, n_mels=128
    )
    # Convert to decibels (log scale, how humans hear)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    return mel_spec_db

print("Deep Learning Setup complete. Spectrogram extractor ready.")

Deep Learning Setup complete. Spectrogram extractor ready.


In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Load the Neural Network
model = tf.keras.models.load_model('instrument_model.keras')

# Load the Label Encoder classes
label_encoder = LabelEncoder()
label_encoder.classes_ = np.load('label_classes.npy', allow_pickle=True)

print("Model and labels successfully loaded! Ready to test.")

Model and labels successfully loaded! Ready to test.


In [13]:
X = []
y_labels = []

data_path = r"E:\ml-instrument detection\data\IRMAS-TrainingData"
window_duration = 1.0  
target_sr = 22050

print("Processing training data into Spectrograms... this will take a few minutes.")

for instrument in os.listdir(data_path):
    folder = os.path.join(data_path, instrument)
    if not os.path.isdir(folder): continue
        
    for file in os.listdir(folder):
        if not file.endswith(".wav"): continue
        
        file_path = os.path.join(folder, file)
        
        try:
            audio, sr = librosa.load(file_path, sr=target_sr)
            samples_per_window = int(window_duration * target_sr)
            
            for i in range(0, len(audio), samples_per_window):
                window = audio[i : i + samples_per_window]
                
                # Pad with zeros if the chunk is slightly too short
                if len(window) < samples_per_window:
                    window = np.pad(window, (0, samples_per_window - len(window)))
                
                spectrogram = extract_mel_spectrogram(window, target_sr)
                
                # Ensure uniform shape (128 mels x 44 time frames for 1 sec)
                if spectrogram.shape[1] == 44: 
                    X.append(spectrogram)
                    y_labels.append(instrument)

        except Exception as e:
            pass 

X = np.array(X)
y_labels = np.array(y_labels)

# Reshape X to add a "color channel" dimension required by CNNs: (Samples, Height, Width, Channels)
X = X[..., np.newaxis] 

# Convert text labels ('gac', 'pia') into integer categories (0, 1, 2...)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_labels)
y_categorical = to_categorical(y_encoded)

print(f"Dataset shape for CNN: {X.shape}")
print("Classes identified:", label_encoder.classes_)

Processing training data into Spectrograms... this will take a few minutes.
Dataset shape for CNN: (17302, 128, 44, 1)
Classes identified: ['dru' 'flu' 'gac' 'gel' 'pia' 'sax' 'tru' 'vio' 'voi']


In [17]:
# 1. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_categorical, test_size=0.2, random_state=42, stratify=y_categorical
)

input_shape = X_train.shape[1:] # Should be (128, 44, 1)
num_classes = len(label_encoder.classes_)

# 2. Build the Convolutional Neural Network (CNN)
model = models.Sequential([
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.BatchNormalization(),
    
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.BatchNormalization(),
    
    layers.Conv2D(128, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.BatchNormalization(),
    
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5), # Prevents overfitting
    layers.Dense(num_classes, activation='softmax') # Outputs probabilities
])

model.compile(optimizer='adam', 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])

# Stop training early if validation accuracy stops improving
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=5, restore_best_weights=True
)

print("Training CNN Model...")
history = model.fit(
    X_train, y_train, 
    epochs=30, 
    batch_size=32, 
    validation_data=(X_test, y_test),
    callbacks=[early_stopping]
)

# 4. Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nFinal CNN Validation Accuracy: {test_acc:.4f}")

Training CNN Model...
Epoch 1/30
433/433 ━━━━━━━━━━━━━━━━━━━━ 57s 123ms/step - accuracy: 0.4058 - loss: 1.7709 - val_accuracy: 0.3597 - val_loss: 2.0465
Epoch 2/30
433/433 ━━━━━━━━━━━━━━━━━━━━ 53s 123ms/step - accuracy: 0.5014 - loss: 1.4114 - val_accuracy: 0.4866 - val_loss: 1.4851
Epoch 3/30
433/433 ━━━━━━━━━━━━━━━━━━━━ 53s 121ms/step - accuracy: 0.5536 - loss: 1.2669 - val_accuracy: 0.5146 - val_loss: 1.4104
Epoch 4/30
433/433 ━━━━━━━━━━━━━━━━━━━━ 55s 127ms/step - accuracy: 0.6067 - loss: 1.1216 - val_accuracy: 0.4975 - val_loss: 1.6670
Epoch 5/30
433/433 ━━━━━━━━━━━━━━━━━━━━ 58s 133ms/step - accuracy: 0.6459 - loss: 1.0200 - val_accuracy: 0.5862 - val_loss: 1.2048
Epoch 6/30
433/433 ━━━━━━━━━━━━━━━━━━━━ 57s 132ms/step - accuracy: 0.6831 - loss: 0.9099 - val_accuracy: 0.6336 - val_loss: 1.0803
Epoch 7/30
433/433 ━━━━━━━━━━━━━━━━━━━━ 56s 130ms/step - accuracy: 0.7221 - loss: 0.7987 - val_accuracy: 0.6177 - val_loss: 1.1560
Epoch 8/30
433/433 ━━━━━━━━━━━━━━━━━━━━ 57s 132ms/step - accu

In [14]:
import numpy as np

# Save the Neural Network
model.save('instrument_model.keras')

# Save the Label Encoder classes
np.save('label_classes.npy', label_encoder.classes_)
print("FILES SUCCESSFULLY SAVED TO HARD DRIVE!")

FILES SUCCESSFULLY SAVED TO HARD DRIVE!


In [15]:
import math
import numpy as np
import librosa
from collections import Counter

test_file = "test.wav"
print(f"Analyzing {test_file} with Deep Learning (Excluding 'gel')...\n")

audio_test, sr_test = librosa.load(test_file, sr=22050)

window_size_sec = 1.0  
hop_size_sec = 0.5     

samples_per_window = int(window_size_sec * sr_test)
hop_length_samples = int(hop_size_sec * sr_test)

time_templates = []
vote_threshold = 0.40 

for start_sample in range(0, len(audio_test) - samples_per_window, hop_length_samples):
    end_sample = start_sample + samples_per_window
    window = audio_test[start_sample:end_sample]
    
    if len(window) < samples_per_window:
        window = np.pad(window, (0, samples_per_window - len(window)))
        
    start_time = start_sample / sr_test
    end_time = end_sample / sr_test
    
    # Extract spectrogram and shape it for the CNN: (1, 128, 44, 1)
    spectrogram = extract_mel_spectrogram(window, sr_test)
    if spectrogram.shape[1] != 44: continue
    
    features_cnn = spectrogram[np.newaxis, ..., np.newaxis] 
    
    # Get all probabilities for this specific 1-second window
    probabilities = model.predict(features_cnn, verbose=0)[0]
    
    # Sort the predictions from highest probability to lowest
    top_indices = np.argsort(probabilities)[::-1]
    
    # Identify the absolute best guess
    best_idx = top_indices[0]
    predicted_instrument = label_encoder.inverse_transform([best_idx])[0]
    confidence = probabilities[best_idx]
    
    # --- NEW EXCLUSION LOGIC ---
    # If the top guess is 'gel', fallback to the second-best guess
    #if predicted_instrument == 'gel':
      #  second_best_idx = top_indices[1]
     #   predicted_instrument = label_encoder.inverse_transform([second_best_idx])[0]
     #   confidence = probabilities[second_best_idx] # Use the lower confidence of the runner-up
    
    # Only record if the model meets the threshold
    if confidence > vote_threshold:
        time_templates.append({
            "start": round(start_time, 1),
            "end": round(end_time, 1),
            "instrument": predicted_instrument,
            "confidence": round(confidence, 2)
        })

# print("--- FINE-GRAINED DETECTION ---")
# for t in time_templates:
#     print(f"[{t['start']}s - {t['end']}s] : {t['instrument'].upper()} (Conf: {t['confidence']})")

print("\n--- 10-SECOND BLOCK SUMMARIES ---")
total_duration = len(audio_test) / sr_test
block_size = 5.0

for block_start in range(0, math.ceil(total_duration), int(block_size)):
    block_end = block_start + block_size
    
    # Gather all valid predictions (keeping the whole dictionary to access confidence)
    valid_predictions = [
        t for t in time_templates
        if block_start <= t['start'] < block_end
    ]
    
    if valid_predictions:
        # Dictionary to store both the count and the accumulated confidence
        stats = {}
        for p in valid_predictions:
            inst = p['instrument']
            if inst not in stats:
                stats[inst] = {'count': 0, 'total_conf': 0.0}
            
            stats[inst]['count'] += 1
            stats[inst]['total_conf'] += p['confidence']
            
        # Sort the instruments: FIRST by count (highest to lowest), THEN by total_conf (highest to lowest)
        sorted_stats = sorted(
            stats.items(), 
            key=lambda item: (item[1]['count'], item[1]['total_conf']), 
            reverse=True
        )
        
        # The winner is now guaranteed to be the first item in the sorted list
        top_instrument = sorted_stats[0][0]
        top_count = sorted_stats[0][1]['count']
        
        print(f"[{int(block_start):02d}s - {int(block_end):02d}s] : {top_instrument.upper()} (Detected {top_count} times)")
    else:
        print(f"[{int(block_start):02d}s - {int(block_end):02d}s] : NO CLEAR INSTRUMENT")

Analyzing test.wav with Deep Learning (Excluding 'gel')...


--- 10-SECOND BLOCK SUMMARIES ---
[00s - 05s] : PIA (Detected 10 times)
[05s - 10s] : PIA (Detected 10 times)
[10s - 15s] : PIA (Detected 5 times)
[15s - 20s] : VOI (Detected 10 times)
[20s - 25s] : VOI (Detected 10 times)
[25s - 30s] : VOI (Detected 9 times)
[30s - 35s] : VOI (Detected 9 times)
[35s - 40s] : VOI (Detected 4 times)
[40s - 45s] : VOI (Detected 9 times)
[45s - 50s] : VOI (Detected 6 times)
[50s - 55s] : SAX (Detected 6 times)
[55s - 60s] : SAX (Detected 7 times)
[60s - 65s] : SAX (Detected 6 times)
[65s - 70s] : VIO (Detected 7 times)
[70s - 75s] : VIO (Detected 4 times)
[75s - 80s] : SAX (Detected 9 times)
[80s - 85s] : SAX (Detected 5 times)
[85s - 90s] : SAX (Detected 4 times)
[90s - 95s] : GEL (Detected 9 times)
[95s - 100s] : SAX (Detected 5 times)
[100s - 105s] : GAC (Detected 9 times)
[105s - 110s] : VOI (Detected 5 times)
[110s - 115s] : FLU (Detected 4 times)
[115s - 120s] : FLU (Detected 7 times)
[120